# H9 · Within-country changes: do firms do better in the years their economy does better?

> **Hypothesis H9:** Within the same company, ROA is higher in years when the national economy grows faster.

| | |
|---|---|
| **Why we ask** | Notebook 03 compares *countries* with each other. That mixes country conditions with everything else that differs between countries (history, culture, tax, listing rules). Comparing a company with **itself in other years** removes all of that: only things that change over time can drive the result. |
| **Prediction** | **Positive** association between same-year national GDP growth and the company's own ROA (firm fixed effects, year fixed effects). |
| **Data** | Yahoo company-years (fiscal 2021 onwards), companies with at least two years. Each variable has the company's own average subtracted (*within* transformation); fiscal-year dummies remove shocks common to all countries. |
| **Primary test (fixed before looking at the result)** | `ROA (pp), company-demeaned ~ GDP growth same year (pp), company-demeaned + fiscal year`, errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Why this is a different question
Most country conditions barely change in the four years we observe. If a variable hardly moves *within* a country, a within-country test has nothing to work with. First check how much of each variable's variation is within vs between countries.

In [2]:
cy = fy.drop_duplicates(["iso3", "fiscal_year"]).set_index(["iso3", "fiscal_year"])
vars_ = {"development index (t-1)": "dev_z", "institutions index (t-1)": "inst_z", "GDP growth, same year (%)": "gdp_growth_now", "inflation, same year (%)": "inflation_now",
         "unemployment (t-1, SD)": "unemployment_z", "private credit to GDP (t-1, SD)": "private_credit_gdp_z"}
rows = []
for lab, c in vars_.items():
    s = cy[c].dropna()
    within = s - s.groupby(level=0).transform("mean")
    rows.append({"variable": lab, "total SD": s.std(), "within-country SD": within.std(), "share of variance within countries": within.var() / s.var()})
vv = pd.DataFrame(rows).set_index("variable")
fig = px.bar(vv.reset_index(), x="variable", y="share of variance within countries", title="How much of each country variable moves within a country over 2021–2025?")
fig.update_layout(height=380, yaxis_tickformat=".0%"); fig.show()
vv

,total SD,within-country SD,share of variance within countries
variable,,,
development index (t-1),0.968,0.094,0.009
institutions index (t-1),0.993,0.068,0.005
"GDP growth, same year (%)",3.035,2.425,0.638
"inflation, same year (%)",3.923,3.193,0.662
"unemployment (t-1, SD)",1.030,0.181,0.031
"private credit to GDP (t-1, SD)",1.007,0.122,0.015


Slow variables (development, institutions) are almost fixed within a country, so a within-country test cannot say anything about them. Fast variables (GDP growth, inflation) do move, so they are the ones we can test.

## 2 · Primary test
Both variables have the company's own mean removed, so every company is compared only with itself.

In [3]:
d = fy.dropna(subset=["roa_pp", "gdp_growth_now"]).copy()
d["n_years"] = d.groupby("company_id").roa_pp.transform("size")
d = d[d.n_years >= 2].copy()
for c in ["roa_pp", "gdp_growth_now", "inflation_now", "growth_pp"]:
    d[c + "_dm"] = d[c] - d.groupby("company_id")[c].transform("mean")
primary = wild_cluster("gdp_growth_now_dm", d, outcome="roa_pp_dm", controls="C(fiscal_year)", B=1999)
print(f"{d.company_id.nunique():,} companies with ≥2 years, {len(d):,} company-years, {primary['clusters']} countries.")
print(f"+1 pp national GDP growth → {primary['coef']:+.3f} pp of the company's own ROA (95% CI {primary['ci_low']:+.3f} to {primary['ci_high']:+.3f}).")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f}")

2,563 companies with ≥2 years, 7,689 company-years, 34 countries.
+1 pp national GDP growth → -0.038 pp of the company's own ROA (95% CI -0.154 to +0.077).
p (cluster-robust) = 0.5211 | p (wild bootstrap) = 0.5690


### Within, between and pooled
The same relationship estimated three ways. If the *within* estimate differs from the *between* one, comparing countries and comparing years tell different stories.

In [4]:
def beta(x, y, data):
    r = fit_cluster(f"{y} ~ {x} + C(fiscal_year)", data)
    return r.params[x], r.params[x] - 1.96 * r.bse[x], r.params[x] + 1.96 * r.bse[x], r.pvalues[x]
dd = d.copy()
dd["gdp_growth_cm"] = dd.groupby("iso3").gdp_growth_now.transform("mean"); dd["roa_cm"] = dd.groupby("iso3").roa_pp.transform("mean")
dd["gdp_growth_cd"] = dd.gdp_growth_now - dd.gdp_growth_cm; dd["roa_cd"] = dd.roa_pp - dd.roa_cm
rows = []
for lab, x, y in [("pooled (all variation)", "gdp_growth_now", "roa_pp"), ("between countries (country means)", "gdp_growth_cm", "roa_pp"),
                  ("within country (country demeaned)", "gdp_growth_cd", "roa_cd"), ("within company (primary)", "gdp_growth_now_dm", "roa_pp_dm")]:
    b, lo, hi, p = beta(x, y, dd)
    rows.append({"estimate": lab, "coef (pp ROA per +1 pp GDP growth)": b, "ci_low": lo, "ci_high": hi, "p (cluster)": p})
est = pd.DataFrame(rows).set_index("estimate")
fig = px.scatter(est.reset_index(), x="coef (pp ROA per +1 pp GDP growth)", y="estimate", error_x=est.ci_high - est.iloc[:, 0], error_x_minus=est.iloc[:, 0] - est.ci_low, title="GDP growth → ROA, estimated four ways")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=360, yaxis_autorange="reversed"); fig.show()
est

,coef (pp ROA per +1 pp GDP growth),ci_low,ci_high,p (cluster)
estimate,,,,
pooled (all variation),0.352,0.044,0.661,0.025
between countries (country means),0.541,0.190,0.893,0.003
within country (country demeaned),-0.047,-0.165,0.072,0.441
within company (primary),-0.038,-0.154,0.077,0.517


## 3 · Which time-varying conditions matter within a company?
Same within-company test for each fast-moving condition (and the slow ones for reference). All tested together, Benjamini–Hochberg corrected.

In [5]:
cands = {"GDP growth, same year (pp)": "gdp_growth_now", "inflation, same year (pp)": "inflation_now", "unemployment (t-1, SD)": "unemployment_z",
         "private credit to GDP (t-1, SD)": "private_credit_gdp_z", "GDP per person (t-1, SD)": "gdp_per_capita_ppp_z", "development index (t-1, SD)": "dev_z", "institutions index (t-1, SD)": "inst_z"}
rows = []
for lab, c in cands.items():
    t = fy.dropna(subset=["roa_pp", c]).copy()
    t = t[t.groupby("company_id").roa_pp.transform("size") >= 2]
    for v in ["roa_pp", c]:
        t[v + "_dm"] = t[v] - t.groupby("company_id")[v].transform("mean")
    r = wild_cluster(c + "_dm", t, outcome="roa_pp_dm", controls="C(fiscal_year)", B=999)
    rows.append({"condition": lab, "coef (pp ROA)": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "firm-years": r["n"]})
allw = pd.DataFrame(rows).set_index("condition"); allw["q (BH)"] = bh(allw["p (wild)"])
allw

,coef (pp ROA),ci_low,ci_high,p (wild),firm-years,q (BH)
condition,,,,,,
"GDP growth, same year (pp)",-0.038,-0.154,0.077,0.571,7689,0.936
"inflation, same year (pp)",-0.003,-0.098,0.091,0.936,7689,0.936
"unemployment (t-1, SD)",-0.225,-1.587,1.137,0.790,10216,0.936
"private credit to GDP (t-1, SD)",0.617,-1.547,2.781,0.630,9427,0.936
"GDP per person (t-1, SD)",2.707,-0.748,6.162,0.214,10216,0.936
"development index (t-1, SD)",0.908,-1.116,2.932,0.490,10216,0.936
"institutions index (t-1, SD)",0.319,-2.119,2.757,0.819,10216,0.936


## 4 · Do countries whose economy accelerated see their firms' returns rise?
A between-country version of the same idea: the change in each country's typical ROA (last two fiscal years vs first two) against the change in its GDP growth over the same years.

In [6]:
yrs = [int(y) for y in sorted(fy.fiscal_year.unique()) if y <= 2025]
early, late = yrs[:2], yrs[-2:]
def med(d, ys):
    t = d[d.fiscal_year.isin(ys)]
    return t.groupby("iso3").agg(roa=("roa_pp", "median"), g=("gdp_growth_now", "mean"), n=("company_id", "nunique"))
ch = med(fy, late).join(med(fy, early), lsuffix="_late", rsuffix="_early").dropna()
ch = ch[(ch.n_late >= MIN_FIRMS) & (ch.n_early >= MIN_FIRMS)]
ch["Δ ROA (pp)"] = ch.roa_late - ch.roa_early; ch["Δ GDP growth (pp)"] = ch.g_late - ch.g_early
r, p, n = spearman(ch["Δ GDP growth (pp)"], ch["Δ ROA (pp)"])
print(f"Fiscal years {early} → {late}: Spearman ρ between change in GDP growth and change in median ROA = {r:+.2f} (p = {p:.3f}, n = {n} countries)")
px.scatter(ch.reset_index().assign(country=lambda d: d.iso3.map(names)), x="Δ GDP growth (pp)", y="Δ ROA (pp)", text="iso3", hover_name="country",
           title="Change in national GDP growth vs change in median company ROA", height=450).update_traces(textposition="top center")

Fiscal years [2021, 2022] → [2024, 2025]: Spearman ρ between change in GDP growth and change in median ROA = -0.16 (p = 0.383, n = 33 countries)


## 5 · Robustness
The primary within-company test in seven samples.

In [7]:
rob, rob_share = robustness("gdp_growth_now_dm", "roa_pp_dm", 1, controls="C(fiscal_year)", data=d)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="GDP growth → within-company ROA (pp per +1 pp) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (positive) in {rob_share:.0%} of samples.")
rob

Predicted sign (positive) in 29% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,-0.038,-0.154,0.077,0.568,34,7689,False
excluding financials & real estate,-0.034,-0.173,0.104,0.632,34,6070,False
excluding small-sample countries,-0.020,-0.152,0.113,0.764,29,7363,False
"excluding USA, China, Japan",-0.087,-0.207,0.032,0.214,31,6792,False
fiscal years 2023 and later,0.002,-0.121,0.124,0.986,34,5074,True
EU countries only,-0.096,-0.221,0.030,0.206,21,3857,False
non-EU countries only,0.033,-0.139,0.205,0.738,13,3832,True


## 6 · Replication on ESEF
EU companies, a longer window (ESEF filings go back before 2021), no sector.

In [8]:
fe = prep.load_firms(con, country, source="ESEF")
fe = fe[fe.fiscal_year.between(2019, 2025)].dropna(subset=["roa_pp", "gdp_growth_now"]).copy()
fe = fe[fe.groupby("company_id").roa_pp.transform("size") >= 2].copy()
for c in ["roa_pp", "gdp_growth_now"]:
    fe[c + "_dm"] = fe[c] - fe.groupby("company_id")[c].transform("mean")
rep = wild_cluster("gdp_growth_now_dm", fe, outcome="roa_pp_dm", controls="C(fiscal_year)", B=1999)
print(f"ESEF ({rep['n']:,} company-years, {fe.company_id.nunique():,} companies, {rep['clusters']} countries): +1 pp GDP growth → {rep['coef']:+.3f} pp ROA (95% CI {rep['ci_low']:+.3f} to {rep['ci_high']:+.3f}), wild p = {rep['p_wild']:.3f}")

ESEF (4,692 company-years, 1,248 companies, 19 countries): +1 pp GDP growth → +0.216 pp ROA (95% CI +0.083 to +0.348), wild p = 0.032


## 7 · Verdict

In [9]:
v = utils.verdict(primary["coef"], primary["p_wild"], 1, rob_share, primary["clusters"])
rep_same = bool(np.sign(rep["coef"]) == 1)
lines = [
 f"Primary: +1 pp national GDP growth → {primary['coef']:+.3f} pp of the company's own ROA (95% CI {primary['ci_low']:+.3f} to {primary['ci_high']:+.3f}), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 f"Robustness: predicted sign in {rob_share:.0%} of seven samples. ESEF replication: {rep['coef']:+.3f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Slow variables (development, institutions) are almost fixed within a country ({vv.loc['development index (t-1)', 'share of variance within countries']:.0%} and {vv.loc['institutions index (t-1)', 'share of variance within countries']:.0%} of variance within), so they cannot be tested this way.",
 f"Country-level change in GDP growth vs change in median ROA: ρ = {r:+.2f} (n = {n}).",
 f"VERDICT H9: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H9", id="H9", title="Within-company response to national growth", statement="Companies earn higher ROA in years when national GDP growth is higher.",
                  expected_sign=1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp ROA per +1 pp GDP growth (within company)",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.9_within_country_changes.ipynb", notes=lines)

Primary: +1 pp national GDP growth → -0.038 pp of the company's own ROA (95% CI -0.154 to +0.077), wild p = 0.569, 34 countries, 7,689 company-years.
Robustness: predicted sign in 29% of seven samples. ESEF replication: +0.216 pp (wild p = 0.032), same direction.
Slow variables (development, institutions) are almost fixed within a country (1% and 0% of variance within), so they cannot be tested this way.
Country-level change in GDP growth vs change in median ROA: ρ = -0.16 (n = 33).
VERDICT H9: NOT SUPPORTED


### Limits
- Only four to five fiscal years, including the 2022–2023 inflation and energy shocks, so the within-company variation in growth is dominated by a few years.
- Year dummies absorb shocks shared by all countries, so only *differences between countries in the same year* identify the effect.
- GDP growth is a national average; it need not describe the markets a multinational actually sells into.
- Association, not causation: faster growth and higher profits can both follow from a third factor (for example commodity prices).

In [10]:
con.close()